In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import time
from numba import njit, prange

mpl.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Times New Roman'],
    'mathtext.fontset': 'stix',          
    'font.size':         11,
    'axes.labelsize':    12,
    'axes.titlesize':    12,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

@njit(nogil=True, fastmath=True)
def SVIz_Euler_numba(alpha_in, bbt1_in):
    Time = 2000.0
    N = 200000
    h = Time / N
    sqrt_h = np.sqrt(h)

    mu   = 0.03
    ga1  = 0.15
    ga2  = 0.15
    bbt2 = 0.8
    rho1 = 0.5
    rho2 = 0.5
    sg1  = 0.3
    sg2  = 0.46

    mu_x1 = np.log(bbt1_in)
    mu_x2 = np.log(bbt2)

    S  = 0.3
    V  = 0.3
    I  = 0.4
    x1 = mu_x1
    x2 = mu_x2

    I_sum     = 0.0
    count     = 0
    start_avg = int(N * 0.8)

    for k in range(N):
        dw0 = np.random.randn()
        dw1 = np.random.randn()

        x1 = x1 + rho1*(mu_x1 - x1)*h + sg1*dw0*sqrt_h
        x2 = x2 + rho2*(mu_x2 - x2)*h + sg2*dw1*sqrt_h

        beta1_t = np.exp(x1)
        beta2_t = np.exp(x2)

        dS = mu - (mu + alpha_in)*S - beta1_t*S*I
        dV = alpha_in*S - beta2_t*V*I - (mu + ga2)*V
        dI = beta1_t*S*I + beta2_t*V*I - (mu + ga1)*I

        S += dS * h
        V += dV * h
        I += dI * h

        if I < 0: I = 0.0
        if S < 0: S = 0.0
        if V < 0: V = 0.0

        if k >= start_avg:
            I_sum += I
            count += 1

    return I_sum / count


@njit(parallel=True)
def compute_grid_parallel(alpha_flat, beta1_flat):
    n       = len(alpha_flat)
    results = np.zeros(n)
    for i in prange(n):
        results[i] = SVIz_Euler_numba(alpha_flat[i], beta1_flat[i])
    return results


def run_experiment_fast():
    print("Running optimised experiment (Numba + parallel)...")
    grid_size  = 30
    alpha_vals = np.linspace(0.05, 1.0,  grid_size)
    beta1_vals = np.linspace(0.1,  4.0,  grid_size)

    Alpha_Grid, Beta1_Grid = np.meshgrid(alpha_vals, beta1_vals)
    alpha_flat = Alpha_Grid.ravel()
    beta1_flat = Beta1_Grid.ravel()

    start_time      = time.time()
    I_Final_flat    = compute_grid_parallel(alpha_flat, beta1_flat)
    I_Final         = I_Final_flat.reshape(grid_size, grid_size)
    end_time        = time.time()
    print(f"Completed in {end_time - start_time:.4f} s")

    fig, ax = plt.subplots(figsize=(5.5, 4.5))   

    contour = ax.contourf(
        Alpha_Grid, Beta1_Grid, I_Final,
        levels=100, cmap='jet'
    )

    cbar = fig.colorbar(contour, ax=ax, pad=0.02)
    cbar.set_label(
        r'Time-averaged infected proportion $\langle I(t) \rangle$',
        fontsize=11
    )
    cbar.ax.tick_params(labelsize=10)

    try:
        cs = ax.contour(
            Alpha_Grid, Beta1_Grid, I_Final,
            levels=[0.001], colors='black', linewidths=1.2
        )
        ax.clabel(cs, fmt={0.001: 'Elimination threshold'},
                  fontsize=9, inline=True)
    except Exception:
        pass

    ax.set_xlabel(r'Vaccination rate $\alpha$',       fontsize=12)
    ax.set_ylabel(r'Average transmission rate $\bar{\beta}_1$', fontsize=12)
    ax.tick_params(direction='in', which='both')     

    fig.tight_layout()
    fig.savefig('I_Heatmap.pdf', format='pdf', bbox_inches='tight')
    plt.show()


if __name__ == "__main__":
    run_experiment_fast()